# Member 1: Data collection, cleaning, feature engineering and Spark demand aggregation (local)
Run top to bottom. Edit only the CONFIG cell.

In [ ]:
# Run once, then restart the kernel. Needs Java 17+ (PySpark 4); on Windows also winutils.exe + hadoop.dll in %HADOOP_HOME%\bin
# %pip install pyspark pyarrow fsspec      # + s3fs | gcsfs | adlfs only if you upload to the cloud

In [ ]:
import os, sys, json
from functools import reduce
from pathlib import Path
from pyspark.sql import SparkSession, Window, functions as F

# ---------------- CONFIG ----------------
RAW_DIR  = os.getenv("RAW_DIR",  r"C:\Angel\big_data_project_updates\data\raw")
OUT_DIR  = os.getenv("OUT_DIR",  r"C:\Angel\big_data_project_updates\data\processed")
DOCS_DIR = os.getenv("DOCS_DIR", r"C:\Angel\big_data_project_updates\docs")
INTERVAL = 15                                        # 15 or 30 minutes
START = os.getenv("START", "2019-01-01")             # analysis period, END exclusive
END   = os.getenv("END",   "2020-07-01")             # = the 18 files 2019-01 .. 2020-06
DRIVER_MEM = "4g"          # "3g" on an 8 GB laptop, "8g" on 16 GB+
CORES = 4                  # fewer cores = less memory pressure
RESTAGE = False            # True = re-parse the CSVs even if staging already exists
DROP_BAD_PASSENGERS = False   # False: keep trips with passenger_count 0 or >6 (still real pickups), set the value to null
CLOUD_DST = None           # e.g. "s3://my-bucket/taxi" (used in the last cell)

# cleaning thresholds
MAX_ZONE, MAX_PAX, MAX_DIST = 263, 6, 100.0          # TLC zone IDs 1..263 (264/265 = unknown)
MIN_DUR, MAX_DUR = 60, 6 * 3600                      # seconds

# column aliases (case-insensitive)
PICKUP  = ["tpep_pickup_datetime", "lpep_pickup_datetime", "pickup_datetime"]
DROPOFF = ["tpep_dropoff_datetime", "lpep_dropoff_datetime", "dropoff_datetime"]
ZONE    = ["PULocationID", "pickup_location_id"]
PAX, DIST = ["passenger_count"], ["trip_distance"]

# ---------------- ENVIRONMENT ----------------
# os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17"     # uncomment if Java is not found
if os.name == "nt":                                  # Windows: Spark needs winutils.exe to write files
    hh = os.environ.get("HADOOP_HOME", r"C:\hadoop")
    os.environ["HADOOP_HOME"] = hh
    os.environ["PATH"] += os.pathsep + hh + r"\bin"
    if not Path(hh, "bin", "winutils.exe").exists():
        print("WARNING: winutils.exe not found in", hh + r"\bin")
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.makedirs(DOCS_DIR, exist_ok=True)
spark_path = lambda p: p if "://" in p else Path(p).resolve().as_posix()   # C:\a\b -> C:/a/b (no URL-encoding of '+')
OUT = spark_path(OUT_DIR)

## 1. Spark session

In [ ]:
spark = (SparkSession.builder.appName("taxi-member1").master(f"local[{CORES}]")
         .config("spark.driver.memory", DRIVER_MEM)
         .config("spark.sql.session.timeZone", "UTC")      # taxi times are local wall-clock; UTC avoids DST gaps
         .config("spark.sql.shuffle.partitions", "64")
         .config("spark.sql.files.maxPartitionBytes", str(64 * 1024 * 1024))   # small input splits = low memory per task
         .config("spark.driver.maxResultSize", "1g")
         .getOrCreate())
spark

## 2. Load raw CSVs and stage them once as Parquet

In [ ]:
def list_files(raw):
    p = Path(raw)
    fs = [p] if p.is_file() else sorted(f for f in p.rglob("*") if f.suffix.lower() in (".parquet", ".csv"))
    return [str(f) for f in fs if "zone" not in f.name.lower()]      # skip taxi+_zone_lookup.csv

def pick(cols, names):
    low = {c.lower(): c for c in cols}
    return next((low[n.lower()] for n in names if n.lower() in low), None)

def read_raw(f):
    if f.lower().endswith(".parquet"):
        return spark.read.parquet(spark_path(f))
    return spark.read.csv(spark_path(f), header=True)      # all strings: one pass, cast below

def read_one(f):
    df = read_raw(f); c = df.columns
    p, d, z, dist, pax = (pick(c, n) for n in (PICKUP, DROPOFF, ZONE, DIST, PAX))
    if not all([p, d, z, dist]):
        raise ValueError(f"{f}: need pickup/dropoff time, PULocationID, trip_distance. Found: {c}")
    return df.select(
        F.to_timestamp(F.col(p)).alias("pickup_ts"),
        F.to_timestamp(F.col(d)).alias("dropoff_ts"),
        F.col(z).cast("int").alias("zone_id"),
        (F.col(pax).cast("int") if pax else F.lit(None).cast("int")).alias("passenger_count"),
        F.col(dist).cast("double").alias("trip_distance"))

files = list_files(RAW_DIR)
assert files, f"No .csv/.parquet files in {RAW_DIR}"
print(len(files), "files:", Path(files[0]).name, "...", Path(files[-1]).name)

# Parse each CSV once, keep the 5 needed columns, store as compact Parquet. Later steps read this instead of ~10 GB of CSV text.
STAGE = f"{OUT}/staging_raw"
marker = Path(OUT_DIR) / "staging_raw" / "_STAGED_OK"
if RESTAGE or "://" in OUT_DIR or not marker.exists():
    for i, f in enumerate(files):
        read_one(f).write.mode("overwrite" if i == 0 else "append").parquet(STAGE)
        print(f"[{i + 1}/{len(files)}] staged {Path(f).name}")
    if "://" not in OUT_DIR:
        marker.write_text("ok")
else:
    print("staging already done, reusing", STAGE)
raw = spark.read.parquet(STAGE)

## 3. Dataset profile

In [ ]:
prof = raw.agg(F.count("*").alias("rows"),
               F.min("pickup_ts").alias("min_pickup"), F.max("pickup_ts").alias("max_pickup"),
               *[F.sum(F.col(c).isNull().cast("int")).alias(f"nulls_{c}") for c in raw.columns]).first().asDict()
prof["distinct_zones"] = raw.select("zone_id").distinct().count()
# quantiles (1%, 50%, 99%) on a 2% sample: cheap and accurate enough for documentation
smp = raw.sample(0.02, seed=1).withColumn("duration_s", F.unix_timestamp("dropoff_ts") - F.unix_timestamp("pickup_ts"))
prof["quantiles_1_50_99_sample"] = smp.select(
    *[F.expr(f"percentile_approx({c}, array(0.01, 0.5, 0.99))").alias(c)
      for c in ("trip_distance", "duration_s", "passenger_count")]).first().asDict()
monthly = raw.groupBy(F.date_format("pickup_ts", "yyyy-MM").alias("month")).count().orderBy("month")
prof["monthly_rows"] = {r["month"]: r["count"] for r in monthly.collect()}
json.dump(prof, open(Path(DOCS_DIR) / "dataset_profile.json", "w"), indent=2, default=str)
print(json.dumps({k: v for k, v in prof.items() if k != "monthly_rows"}, indent=2, default=str))
# stray months (e.g. 2001, 2008, 2090) are bad timestamps in the source; they are removed by the period filter below
monthly.show(100)

## 4. Clean

In [ ]:
print("period:", START, "->", END)
df = raw.withColumn("duration_s", F.unix_timestamp("dropoff_ts") - F.unix_timestamp("pickup_ts"))
bad = lambda cond: F.coalesce(~cond, F.lit(True))      # NULL counts as invalid
t0, t1 = F.lit(START).cast("timestamp"), F.lit(END).cast("timestamp")
pax_bad = F.col("passenger_count").isNotNull() & ~F.col("passenger_count").between(1, MAX_PAX)

rules = {
    "missing_required": F.col("pickup_ts").isNull() | F.col("dropoff_ts").isNull() | F.col("zone_id").isNull(),
    "pickup_outside_period": bad((F.col("pickup_ts") >= t0) & (F.col("pickup_ts") < t1)),
    "invalid_duration": bad(F.col("duration_s").between(MIN_DUR, MAX_DUR)),
    "invalid_distance": bad((F.col("trip_distance") > 0) & (F.col("trip_distance") <= MAX_DIST)),
    "invalid_zone": bad(F.col("zone_id").between(1, MAX_ZONE)),
}
if DROP_BAD_PASSENGERS:
    rules["invalid_passengers"] = pax_bad
df = df.select("*", *[v.alias(k) for k, v in rules.items()])
any_bad = reduce(lambda a, b: a | b, [F.col(k) for k in rules])

stats = df.agg(F.count("*").alias("total_rows"),
               F.sum(F.col("passenger_count").isNull().cast("int")).alias("passenger_count_null"),
               F.sum(pax_bad.cast("int")).alias("passenger_count_out_of_range"),
               *[F.sum(F.col(k).cast("int")).alias(k) for k in rules],
               F.sum(any_bad.cast("int")).alias("removed_invalid")).first().asDict()
print(json.dumps(stats, indent=2))

valid = (df.where(~any_bad)
           .dropDuplicates(["pickup_ts", "dropoff_ts", "zone_id", "trip_distance", "passenger_count"])
           .drop(*rules)
           .withColumn("passenger_count", F.when(F.col("passenger_count").between(1, MAX_PAX), F.col("passenger_count"))))

## 5. Trip-level features and cleaned dataset

In [ ]:
trips = (valid.withColumn("pickup_year", F.year("pickup_ts"))
              .withColumn("pickup_month", F.month("pickup_ts"))
              .withColumn("pickup_day", F.dayofmonth("pickup_ts"))
              .withColumn("pickup_hour", F.hour("pickup_ts"))
              .withColumn("pickup_weekday", (F.dayofweek("pickup_ts") + 5) % 7)      # Mon=0 .. Sun=6
              .withColumn("is_weekend", (F.col("pickup_weekday") >= 5).cast("int")))

lookup = next(Path(RAW_DIR).rglob("*zone*lookup*.csv"), None)                        # matches taxi+_zone_lookup.csv
print("zone lookup:", lookup)
if lookup:
    zones = spark.read.csv(spark_path(str(lookup)), header=True, inferSchema=True).select(
        F.col("LocationID").cast("int").alias("zone_id"),
        F.col("Borough").alias("borough"), F.col("Zone").alias("zone_name"))
    trips = trips.join(F.broadcast(zones), "zone_id", "left").fillna({"borough": "Unknown", "zone_name": "Unknown"})
else:
    trips = trips.withColumn("borough", F.lit("Unknown")).withColumn("zone_name", F.lit("Unknown"))

trips_path = f"{OUT}/trips_clean"
trips.write.mode("overwrite").partitionBy("pickup_year", "pickup_month").parquet(trips_path)
trips = spark.read.parquet(trips_path)
stats["rows_clean"] = trips.count()                  # counted on the written Parquet (cheap, no re-run of the cleaning)
stats["duplicates_removed"] = stats["total_rows"] - stats["removed_invalid"] - stats["rows_clean"]
print("clean trips:", stats["rows_clean"], "| duplicates removed:", stats["duplicates_removed"])
trips.groupBy("pickup_year", "pickup_month").count().orderBy("pickup_year", "pickup_month").show(24)

## 6. Demand aggregation (zone x interval) with lag features

In [ ]:
sec = INTERVAL * 60
agg = (trips.withColumn("interval_start", F.timestamp_seconds(F.floor(F.unix_timestamp("pickup_ts") / sec) * sec))
            .groupBy("interval_start", "zone_id").agg(F.count("*").alias("demand")))

lo, hi = agg.agg(F.min(F.unix_timestamp("interval_start")), F.max(F.unix_timestamp("interval_start"))).first()
times = spark.range((hi - lo) // sec + 1).select(
    F.timestamp_seconds(F.lit(lo) + F.col("id") * sec).alias("interval_start"))
zone_ids = agg.select("zone_id").distinct()
# full interval x zone grid: zero-demand intervals exist and lag offsets are exact
demand = times.crossJoin(zone_ids).join(agg, ["interval_start", "zone_id"], "left").fillna({"demand": 0})

spd = 1440 // INTERVAL                                    # intervals per day
w = Window.partitionBy("zone_id").orderBy("interval_start")
for name, k in {"lag_1": 1, "lag_2": 2, "lag_4": 4, "lag_1d": spd, "lag_1w": 7 * spd}.items():
    demand = demand.withColumn(name, F.lag("demand", k).over(w))
demand = (demand
    .withColumn("roll_mean_4", F.avg("demand").over(w.rowsBetween(-4, -1)))
    .withColumn("roll_mean_1d", F.avg("demand").over(w.rowsBetween(-spd, -1)))
    .withColumn("hour", F.hour("interval_start"))
    .withColumn("slot_of_day", ((F.hour("interval_start") * 60 + F.minute("interval_start")) / INTERVAL).cast("int"))
    .withColumn("day", F.dayofmonth("interval_start"))
    .withColumn("weekday", (F.dayofweek("interval_start") + 5) % 7)
    .withColumn("month", F.month("interval_start"))
    .withColumn("is_weekend", (F.col("weekday") >= 5).cast("int")))

demand_path = f"{OUT}/demand_{INTERVAL}min"
demand.coalesce(8).write.mode("overwrite").parquet(demand_path)
demand = spark.read.parquet(demand_path)
stats["demand_rows"] = demand.count()
print("intervals:", (hi - lo) // sec + 1, "| zones:", zone_ids.count(), "| rows:", stats["demand_rows"])
demand.orderBy("zone_id", "interval_start").show(5)

## 7. Checks and quality report

In [ ]:
assert demand.groupBy("interval_start", "zone_id").count().where("count > 1").count() == 0, "duplicate keys"
assert demand.agg(F.sum("demand")).first()[0] == stats["rows_clean"], "demand total != clean trips"

stats.update(period_start=START, period_end=END, interval_min=INTERVAL)
json.dump(stats, open(Path(DOCS_DIR) / "quality_report.json", "w"), indent=2, default=str)
print(json.dumps(stats, indent=2, default=str))
demand.groupBy("hour").agg(F.sum("demand").alias("pickups")).orderBy("hour").show(24)

## 8. Upload to cloud storage (optional)

In [ ]:
# pip install s3fs | gcsfs | adlfs ; credentials via env vars / CLI login. Set CLOUD_DST in the config cell.
if CLOUD_DST:
    import fsspec
    fs = fsspec.filesystem(CLOUD_DST.split("://")[0])
    for src, sub in [(RAW_DIR, "raw"), (OUT_DIR, "processed")]:
        dst = f"{CLOUD_DST.rstrip('/')}/{sub}/"
        fs.put(src.rstrip("\\/") + "/", dst, recursive=True)
        print("uploaded", src, "->", dst, "|", len(fs.find(dst)), "files")
else:
    print("CLOUD_DST not set, skipping upload")